In [148]:
import pygod
from pygod.detector import DOMINANT 
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import from_networkx
from sklearn.metrics import roc_auc_score, f1_score

import pandas as pd
import torch 
from pathlib import Path
import pickle 
import numpy as np 
import random
import math
seed = 21


random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)




device = 'cpu'

## Data preparation

In [149]:
!cd ../data/russia && ls

0.7_datasets.pkl                 0.7_datasets.pkl_0.999U
0.7_datasets.pkl_0.5U            0.7_datasets.pkl_0.99U
0.7_datasets.pkl_0.75U           0.7_datasets.pkl_0.9U
0.7_datasets.pkl_0.95U           sbert_nodeattributes_mostPop5.pt


In [150]:
data_path = Path("../data/russia")
graph_data_file = "0.7_datasets.pkl"

with open(data_path / graph_data_file,"rb") as file:
    graph_data = pickle.load(file)
graph_data.keys()

dict_keys(['graph', 'coRT', 'coURL', 'hashSeq', 'fastRT', 'tweetSim', 'labels', 'splits'])

In [151]:
embedding_file = "sbert_nodeattributes_mostPop5.pt"
embeddings = torch.load(data_path / embedding_file, map_location=device)
type(embeddings),embeddings.shape

(torch.Tensor, torch.Size([716, 768]))

In [152]:
graph = graph_data['graph']

degrees = np.array([
    graph.degree(node)
    for node in range(graph.number_of_nodes())
])

degree_bins = np.array([
    0 if degree <= 1 else math.ceil(math.log2(degree))
    for degree in degrees
])


degree_features = torch.nn.functional.one_hot(
    torch.tensor(degree_bins),
    num_classes=int(degree_bins.max()) + 1,
).float()

# 768 SBERT features + 9 degree features = 777
features = torch.cat([
    embeddings.float(),
    degree_features,
], dim=1)



In [153]:
data = from_networkx(graph)
data.x = features
data.y = torch.tensor(graph_data["labels"], dtype=torch.long)



## Hyperparameters and modeling

In [154]:
!cd ../artifacts/results/benchmark/ && ls

UAE
all_models_best_auc_by_dataset.csv
asgae_asgae_only_grid
asgae_remaining_grid
asgae_smoke
best_auc_tables
best_threshold_f1_by_dataset_model.csv
best_threshold_f1_by_dataset_model_split.csv
china
cuba
gadnr_auc_logs
gadnr_auc_rerun_logs
gadnr_best_auc_by_dataset.csv
gal_contrastive_best
gal_contrastive_logs
iran
repeated_seed_best_auc
repeated_seed_best_auc_tables
russia
saved_models_best_auc
seeded_model_score_thresholds
seeded_model_scores
venezuela


In [155]:
hp_path = Path("../artifacts/results/benchmark/")
hp_df = pd.read_csv(hp_path / "all_models_best_auc_by_dataset.csv")
hp_df = hp_df[(hp_df['dataset'] == 'russia') & (hp_df['model'] == 'pygod_dominant')]
hp_df = hp_df.dropna(axis=1)
hp_df.head()

,dataset,model,auc,macro_f1,threshold,runtime_seconds,model_size_mb,num_parameters,epochs,hid_dim,lr
3,russia,pygod_dominant,0.856904,0.721194,0.5,0.797901,0.43021,112777.0,100.0,64.0,0.004


In [156]:
Path("../artifacts/results/benchmark/")
hp_df = pd.read_csv(hp_path / "all_models_best_auc_by_dataset.csv")
hp_df = hp_df[(hp_df['dataset'] == 'russia') & (hp_df['model'] == 'pygod_dominant')]
hp_df = hp_df.dropna(axis=1)
hp_df.head()

,dataset,model,auc,macro_f1,threshold,runtime_seconds,model_size_mb,num_parameters,epochs,hid_dim,lr
3,russia,pygod_dominant,0.856904,0.721194,0.5,0.797901,0.43021,112777.0,100.0,64.0,0.004


In [157]:


hp = hp_df.iloc[0]

model = DOMINANT(
    hid_dim=int(hp["hid_dim"]),
    num_layers=2,                 # Not included in hp_df
    epoch=int(hp["epochs"]),      # CSV column is "epochs"; DOMINANT uses "epoch"
    lr=float(hp["lr"]),
    gpu=-1,                       # CPU
)

model

DOMINANT(act=<function relu at 0x137ea91c0>,
         backbone=<class 'torch_geometric.nn.models.basic_gnn.GCN'>,
         batch_size=0, compile_model=False, contamination=0.1, dropout=0.0,
         epoch=100, gpu=None, hid_dim=64, lr=0.004, num_layers=2,
         num_neigh=[-1, -1], save_emb=False, sigmoid_s=False, verbose=0,
         weight=0.5, weight_decay=0.0)

## Train

Traing with the entire graph + norm mim-max anorm score + 0.5 treshold

In [158]:
labels = data.y.cpu().numpy()
threshold = 0.5

for seed in range(10):
    # Set seed before creating the model
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Fresh model with new initial weights
    model = DOMINANT(
        hid_dim=int(hp["hid_dim"]),
        num_layers=4,
        epoch=int(hp["epochs"]),
        lr=float(hp["lr"]),
        batch_size=0,
        num_neigh=-1,
        gpu=-1,
    )

    # Train on the complete graph
    model.fit(data)

    # Training-data anomaly scores
    scores = model.decision_score_.detach().cpu().numpy()

    # Min-max normalization
    normalized_scores = (
        (scores - scores.min())
        / (scores.max() - scores.min() + 1e-12)
    )

    predictions = (normalized_scores > threshold).astype(int)

    auc = roc_auc_score(labels, normalized_scores)
    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    print(
        f"Seed {seed} | "
        f"AUC: {auc:.4f} | "
        f"Macro-F1: {macro_f1:.4f}"
    )

Seed 0 | AUC: 0.8480 | Macro-F1: 0.7159
Seed 1 | AUC: 0.8608 | Macro-F1: 0.7190
Seed 2 | AUC: 0.7937 | Macro-F1: 0.4143
Seed 3 | AUC: 0.7448 | Macro-F1: 0.4033
Seed 4 | AUC: 0.7511 | Macro-F1: 0.4077
Seed 5 | AUC: 0.8440 | Macro-F1: 0.7068
Seed 6 | AUC: 0.8355 | Macro-F1: 0.7068
Seed 7 | AUC: 0.8502 | Macro-F1: 0.7112
Seed 8 | AUC: 0.8359 | Macro-F1: 0.7068
Seed 9 | AUC: 0.8328 | Macro-F1: 0.4663


In [180]:
model = DOMINANT(
    hid_dim=int(hp["hid_dim"]),
    num_layers=2,
    epoch=int(hp["epochs"]),
    lr=float(hp["lr"]),
    batch_size=0,
    num_neigh=-1,
    gpu=-1,
)

# Train once on the complete graph
model.fit(data)

scores = model.decision_score_.detach().cpu().numpy()

# Normalize once using all graph scores
scores = (
    (scores - scores.min())
    / (scores.max() - scores.min() + 1e-12)
)

val_macro_f1_values = []
val_auc_values = []


# Use the same model and scores for all five splits
for split_id in range(5):
    split = graph_data["splits"][split_id]

    train_mask = np.asarray(split["train"], dtype=bool)
    val_mask = np.asarray(split["val"], dtype=bool)

    # Try every possible threshold found in the training scores
    unique_scores = np.unique(scores[train_mask])

    thresholds = np.concatenate([
        [np.nextafter(unique_scores.min(), -np.inf)],
        unique_scores,
        [np.nextafter(unique_scores.max(), np.inf)],
    ])

    # Select the threshold using training macro-F1
    best_threshold = 0.5
    best_train_macro_f1 = -1

    for threshold in thresholds:
        train_pred = scores[train_mask] > threshold

        train_macro_f1 = f1_score(
            labels[train_mask],
            train_pred,
            average="macro",
            zero_division=0,
        )

        if train_macro_f1 > best_train_macro_f1:
            best_train_macro_f1 = train_macro_f1
            best_threshold = threshold

    # Evaluate the selected threshold on validation data
    val_pred = scores[val_mask] > best_threshold

    val_macro_f1 = f1_score(
        labels[val_mask],
        val_pred,
        average="macro",
        zero_division=0,
    )

    val_auc = roc_auc_score(
        labels[val_mask],
        scores[val_mask],
    )

    val_macro_f1_values.append(val_macro_f1)
    val_auc_values.append(val_auc)

    print(f"\nSplit {split_id}")
    print(f"Best training threshold: {best_threshold:.4f}")
    print(f"Train Macro-F1: {best_train_macro_f1:.4f}")
    print(f"Validation Macro-F1: {val_macro_f1:.4f}")
    print(f"Validation AUC: {val_auc:.4f}")


Split 0
Best training threshold: 0.2884
Train Macro-F1: 0.7438
Validation Macro-F1: 0.7812
Validation AUC: 0.8572

Split 1
Best training threshold: 0.2985
Train Macro-F1: 0.7724
Validation Macro-F1: 0.7516
Validation AUC: 0.8554

Split 2
Best training threshold: 0.2884
Train Macro-F1: 0.7664
Validation Macro-F1: 0.7760
Validation AUC: 0.8614

Split 3
Best training threshold: 0.2993
Train Macro-F1: 0.7468
Validation Macro-F1: 0.7851
Validation AUC: 0.8434

Split 4
Best training threshold: 0.2884
Train Macro-F1: 0.7702
Validation Macro-F1: 0.7361
Validation AUC: 0.8583


In [179]:
print("\nResults across five splits")

print(
    f"Validation Macro-F1: "
    f"{np.mean(val_macro_f1_values):.4f} ± "
    f"{np.std(val_macro_f1_values, ddof=1):.4f}"
)

print(
    f"Validation AUC: "
    f"{np.mean(val_auc_values):.4f} ± "
    f"{np.std(val_auc_values, ddof=1):.4f}"
)


Results across five splits
Validation Macro-F1: 0.7709 ± 0.0409
Validation AUC: 0.8322 ± 0.0422
